# FaceDetector Processing

---

Tahapan ini mencakup pengambilan, pemrosesan dan evaluasi untuk skenario tugas dan istirahat
- Mengambil data dari file npy
- Melakukan pre-processing pada sinyal rPPG dan PPG
- Melakukan extraksi HR dan HRV (SDNN dan RMSSD)
- Evaluasi MAE, RMSE dan PC

In [37]:
## Importing Dependencies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from typing import Optional, Tuple
import seaborn as sns
import neurokit2 as nk
import scipy
import os
from prettytable import PrettyTable
from dataclasses import dataclass
from sklearn.metrics import mean_absolute_error
from scipy.stats import pearsonr
import numpy as np


In [38]:
@dataclass
class SignalData:
    subject: str
    task: str
    method: str  
    raw: np.ndarray
    fs: int
    preprocessed: np.ndarray = None


## Preprocessing & Signal Extraction

### Define the processing method (Preprocess, Ekstraksi HR / HRV Metrik Pipeline)
---

In [39]:
def preprocess_rppg(rppg_signal, fs_rppg=35, fs_ppg=64):
    # Target: exactly 180 seconds at 64 Hz = 11520 samples
    target_duration = 180.0  # seconds
    target_samples = 11520  # Exact target
    
    # Calculate current duration
    current_duration = len(rppg_signal) / fs_rppg
    print(f"Original samples: {len(rppg_signal)}, Current duration: {current_duration:.2f}s")
    
    # Trim to exactly 180 seconds if longer
    if current_duration > target_duration:
        samples_to_keep = int(target_duration * fs_rppg)  # 180 * 35 = 6300 samples
        rppg_trimmed = rppg_signal[:samples_to_keep]
    else:
        rppg_trimmed = rppg_signal
        print(f"No trimming needed, using {len(rppg_trimmed)} samples")
    
    # Cubic spline interpolation with EXACT target samples
    duration_trimmed = len(rppg_trimmed) / fs_rppg
    t_old = np.arange(len(rppg_trimmed)) / fs_rppg
    t_new = np.linspace(0, duration_trimmed, target_samples, endpoint=False)  # endpoint=False is KEY
    cs = scipy.interpolate.CubicSpline(t_old, rppg_trimmed)
    rppg_up = cs(t_new)
    
    print(f"After interpolation: {len(rppg_up)} samples (target: {target_samples})")
    
    # Bandpass filter at new rate
    b, a = scipy.signal.butter(3, [0.7, 2.5], btype='band', fs=fs_ppg)
    rppg_filtered = scipy.signal.filtfilt(b, a, rppg_up)
    
    # Normalize
    rppg_filtered = (rppg_filtered - np.mean(rppg_filtered)) / np.std(rppg_filtered)
    
    # Final verification
    print(f"Final output: {len(rppg_filtered)} samples")
    assert len(rppg_filtered) == target_samples, f"Length mismatch: got {len(rppg_filtered)}, expected {target_samples}"
    
    return rppg_filtered

In [40]:
def preprocess_ppg(ppg_signal, fs_ppg=64):

    # Bandpass filter the PPG signal
    b, a = scipy.signal.butter(3, [0.7, 2.5], btype='band', fs=fs_ppg)
    ppg_filtered = scipy.signal.filtfilt(b, a, ppg_signal)

    # Normalize the signal
    ppg_filtered = (ppg_filtered - np.mean(ppg_filtered)) / np.std(ppg_filtered)

    return ppg_filtered


Main Methods

---

Opening the npy files for FaceDetector

In [41]:
root_path = "dataset_numpy"
subjects = ["s41", "s42", "s43", "s44","s45","s46","s47","s48","s49","s50","s51","s52", "s53","s54","s55","s56"]
tasks = ["T1", "T2", "T3"] # Rest, Task

sample_rate_gt = 64  # Hz
sample_rate_video = 35 # Hz

all_signals = []

for subject in subjects:
    for task in tasks:
        # --- Ground truth PPG ---
        gt_file = os.path.join(root_path, subject, f"bvp_{subject}_{task}.csv")
        gt_data = pd.read_csv(gt_file, header=None).values.flatten()
        gt_signal = preprocess_ppg(gt_data, fs_ppg=sample_rate_gt)

        all_signals.append(
            SignalData(
                subject=subject,
                task=task,
                method="PPG",
                raw=gt_data,
                fs=sample_rate_gt,
                preprocessed=gt_signal
            )
        )

        # --- rPPG methods ---
        for method in ["POS", "LGI", "OMIT", "GREEN", "CHROM"]:
            file_path = os.path.join(root_path, subject, f"{subject}_{task}_{method}_rppg.npy")
            rppg_raw = np.load(file_path)

            rppg_preproc = preprocess_rppg(rppg_raw,
                                           fs_rppg=sample_rate_video,
                                           fs_ppg=sample_rate_gt)

            all_signals.append(
                SignalData(
                    subject=subject,
                    task=task,
                    method=method,
                    raw=rppg_raw,
                    fs=sample_rate_video,
                    preprocessed=rppg_preproc
                )
            )

## Sanity check the pipeline for the result
## Expected all methods should have length of 11520 (64 Hz * 180 sec = 11520 samples long)
summary = pd.DataFrame([{
    "Subject": sig.subject,
    "Task": sig.task,
    "Method": sig.method,
    "RawLen": len(sig.raw),
    "PreprocLen": len(sig.preprocessed)
} for sig in all_signals])

print(summary.head(20))


Original samples: 6325, Current duration: 180.71s
After interpolation: 11520 samples (target: 11520)
Final output: 11520 samples
Original samples: 6325, Current duration: 180.71s
After interpolation: 11520 samples (target: 11520)
Final output: 11520 samples
Original samples: 6325, Current duration: 180.71s
After interpolation: 11520 samples (target: 11520)
Final output: 11520 samples
Original samples: 6325, Current duration: 180.71s
After interpolation: 11520 samples (target: 11520)
Final output: 11520 samples
Original samples: 6300, Current duration: 180.00s
No trimming needed, using 6300 samples
After interpolation: 11520 samples (target: 11520)
Final output: 11520 samples
Original samples: 6325, Current duration: 180.71s
After interpolation: 11520 samples (target: 11520)
Final output: 11520 samples
Original samples: 6325, Current duration: 180.71s
After interpolation: 11520 samples (target: 11520)
Final output: 11520 samples
Original samples: 6325, Current duration: 180.71s
After in

Notes:

---

The signal been in the same length and preprocessed

and it's fine to move on to the HR / HRV Extraction

## Evaluasi Performa rPPG
---

Tahapan ini akan mengukur HR / HRV dari rPPG dan PPG

In [42]:
@dataclass
class MetricData:
    subject: str
    task: str
    method: str
    hr: float
    sdnn: float
    rmssd: float


**Formula Perhitungan HR dan HRV**

**Heart Rate (HR):**
$$
HR = \frac{60}{\overline{RR}}
$$
- $\overline{RR}$ = rata-rata interval RR (detik) antar puncak sinyal

**SDNN (Standard Deviation of NN intervals):**
$$
SDNN = \text{std}(RR_{ms})
$$
- $(RR_{ms})$ = interval RR dalam milidetik

**RMSSD (Root Mean Square of Successive Differences):**
$$
RMSSD = \sqrt{\frac{1}{N-1} \sum_{i=1}^{N-1} (RR_{ms,i+1} - RR_{ms,i})^2}
$$
- $(RR_{ms,i})$ = interval RR ke-i dalam milidetik
- $(N)$ = jumlah interval RR valid

**Langkah:**
1. Deteksi puncak sinyal (peak detection)
2. Hitung interval RR antar puncak: $(RR = \frac{\Delta \text{peak}}{fs})$
3. Bersihkan RR di luar rentang 0.3–2.0 detik
4. Konversi RR ke milidetik: $(RR_{ms} = RR \times 1000)$
5. Hitung HR, SDNN, RMSSD dengan rumus

In [43]:
import numpy as np
import scipy.signal

def compute_metrics(signal, fs, segment_len=25, seg_threshold=0.3):
    
    # 1. Detect peaks
    peaks, _ = scipy.signal.find_peaks(signal, prominence=0.5)
    # peaks, _ = scipy.signal.find_peaks(signal)
    
    if len(peaks) < 2:
        return np.nan, np.nan, np.nan  # Not enough peaks
    
    # 2. Compute RR intervals (s)
    rr = np.diff(peaks) / fs
    rr = np.asarray(rr, dtype=float)
    
    # 3. Hard physiologic filter
    rr = rr[(rr >= 0.3) & (rr <= 2.0)]
    if len(rr) < 2:
        return np.nan, np.nan, np.nan
    
    # # 4. Segment-based ±30% filtering
    # clean_rr = []
    # num_segments = len(rr) // segment_len
    # for i in range(num_segments):
    #     seg = rr[i*segment_len:(i+1)*segment_len]
    #     mean_seg = np.mean(seg)
    #     lower = mean_seg * (1 - seg_threshold)
    #     upper = mean_seg * (1 + seg_threshold)
    #     clean_seg = seg[(seg >= lower) & (seg <= upper)]
    #     clean_rr.extend(clean_seg)
        
    rr_final = np.array(rr)
    if len(rr_final) < 2:
        return np.nan, np.nan, np.nan
    
    # 6. Compute HR
    hr = int(60 / np.mean(rr_final))
    
    # 7. Convert to ms for HRV
    rr_ms = rr_final * 1000
    
    # 8. Compute HRV metrics
    sdnn = int(np.std(rr_ms))
    rmssd = int(np.sqrt(np.mean(np.square(np.diff(rr_ms)))))
    
    return hr, sdnn, rmssd


**Create a Header Template for HR, HRV**

---

Membuat header template agar tabel lebih padat dan bisa memuat banyak informasi, akan ada 3 tabel (HR, SDNN, RMSSD), dikelompokan berdasarkan 
- Tipe skenario
- Metode pengukuran data

In [44]:
arrays = [
    [""] + ["Rest"]*6 + ["Speech"]*6 + ["Task"]*6,
    ["Subject", "GT", "POS", "GREEN", "LGI", "OMIT", "CHROM",
     "GT", "POS", "GREEN", "LGI", "OMIT", "CHROM",
     "GT", "POS", "GREEN", "LGI", "OMIT", "CHROM"]
]


columns = pd.MultiIndex.from_arrays(arrays)

In [45]:
all_metrics = []

for sig in all_signals:
    ## Semua sinyal di resample ke 64 Hz
    hr, sdnn, rmssd = compute_metrics(sig.preprocessed, 64) 
    all_metrics.append(MetricData(sig.subject, sig.task, sig.method, hr, sdnn, rmssd))


In [46]:
df_metrics = pd.DataFrame([{
    "Subject": metrics.subject,
    "Task": metrics.task,  # Keep original values first
    "Method": metrics.method,
    "HR": metrics.hr,
    "SDNN": metrics.sdnn,
    "RMSSD": metrics.rmssd
} for metrics in all_metrics])

# Then remap the Task column
task_mapping = {"T1": "Istirahat", "T2": "Speech", "T3": "Tugas Mental"}
df_metrics["Task"] = df_metrics["Task"].map(task_mapping)

#### Ekstraksi HR

---

In [47]:
rows = []

for subject in subjects:
    row = [subject]
    # Rest (T1) HR
    for method in ["PPG", "POS", "GREEN", "LGI", "OMIT", "CHROM"]:
        val = df_metrics.query("Subject == @subject and Task == 'Istirahat' and Method == @method")["HR"]
        row.append(val.values[0] if len(val) > 0 else None)

    # Task (T2) HR
    for method in ["PPG", "POS", "GREEN", "LGI", "OMIT", "CHROM"]:
        val = df_metrics.query("Subject == @subject and Task == 'Speech' and Method == @method")["HR"]
        row.append(val.values[0] if len(val) > 0 else None)

    # Task (T3) HR
    for method in ["PPG", "POS", "GREEN", "LGI", "OMIT", "CHROM"]:
        val = df_metrics.query("Subject == @subject and Task == 'Tugas Mental' and Method == @method")["HR"]
        row.append(val.values[0] if len(val) > 0 else None)

    rows.append(row)

df_hr = pd.DataFrame(rows, columns=columns)
df_hr

Rest                            Speech                             \
   Subject   GT  POS GREEN  LGI OMIT CHROM     GT  POS GREEN  LGI OMIT CHROM   
0      s41   96   96    88   96   96    95     73  105    76  104  105   104   
1      s42   70   71    76   71   71    71     76   86    71   76   73    83   
2      s43   95   94    88   92   92    93     85   90    68   82   82    87   
3      s44   63   67    67   63   63    67     67   70    65   70   69    72   
4      s45   77   76    74   76   76    78     80   86    64   75   75    85   
5      s46   74   74    74   74   74    74     76   86    74   82   82    84   
6      s47   77   77    72   78   78    78     74  105    68  103  102   104   
7      s48   65   72    65   71   70    71     77   86    70   80   81    84   
8      s49  101  101    87   99   99   101     75   95    65   83   85    92   
9      s50  103  102    89  102  102   102     83  103    80  102  102   103   
10     s51   62   62    67   62   62    63     65   81    61   77   79    80   
11     s52   78   89    82   89   89    88     79   96    74   91   91    94   
12     s53   64   85    82   85   85    85     70   85    66   82   82    85   
13     s54   87   93    76   90   90    92     87   87    73   84   83    86   
14     s55   88   93    64   89   89    91     81   93    72   91   91    92   
15     s56   71   84    68   79   79    80     76   87    68   77   76    83   

   Task                             
     GT  POS GREEN  LGI OMIT CHROM  
0    74   99    74   96   96    97  
1    83   89    73   81   82    85  
2    84   89    70   84   84    87  
3    65   67    66   64   65    71  
4    83   83    69   77   76    80  
5    88   85    84   85   85    86  
6    78   91    62   84   85    90  
7    75   82    69   78   77    83  
8    70   89    77   84   84    89  
9   103  103    81  103  103   100  
10   72   75    60   72   72    74  
11   74   86    76   84   84    84  
12   68   75    73   73   73    75  
13   90   90    75   84   84    89  
14   86   85    67   79   80    80  
15   75   82    76   80   80    84

In [48]:
methods = ["POS", "CHROM", "GREEN", "LGI", "OMIT"]
conditions = ["Rest", "Speech", "Task"]

eval_results = []

for cond in conditions:
    gt = df_hr[(cond, "GT")]  # ground truth for condition rest / task
    for method in methods:
        row=[method]
        pred = df_hr[(cond, method)]
        
        mae = mean_absolute_error(gt, pred)
        rmse = np.sqrt(np.mean((gt - pred) ** 2))
        pc, _ = pearsonr(gt, pred)
        
        eval_results.append({
            "Condition": cond,
            "Method": method,
            "MAE": mae,
            "RMSE": rmse,
            "PC": pc
        })

eval_df_hr = pd.DataFrame(eval_results)
print("HR (satuan: BPM)")
eval_df_hr

HR (satuan: BPM)


,Condition,Method,MAE,RMSE,PC
0,Rest,POS,4.4375,7.335700,0.890876
1,Rest,CHROM,4.1250,6.698881,0.908825
2,Rest,GREEN,7.8750,10.191909,0.698441
3,Rest,LGI,3.6875,6.562202,0.897551
4,Rest,OMIT,3.6250,6.509608,0.898569
5,Speech,POS,13.5625,16.145820,0.357380
6,Speech,CHROM,12.2500,15.024979,0.313260
7,Speech,GREEN,7.1875,8.584142,0.538058
8,Speech,LGI,9.8125,13.362728,0.247141
9,Speech,OMIT,10.2500,13.619838,0.215274


In [49]:
# Calculate average MAE and PC for each method across conditions
methods = ["POS", "GREEN", "LGI", "OMIT", "CHROM"]

print("Average MAE and PC across Rest and Task conditions:")
print("=" * 60)
print(f"{'Method':<10} {'Avg MAE':<12} {'Avg PC':<12}")
print("-" * 60)

for method in methods:
    method_data = eval_df_hr[eval_df_hr["Method"] == method]
    avg_mae = method_data["MAE"].mean()
    avg_pc = method_data["PC"].mean()
    print(f"{method:<10} {avg_mae:<12.4f} {avg_pc:<12.6f}")

print("=" * 60)

# Create a summary DataFrame
summary_avg = pd.DataFrame([{
    "Method": method,
    "Avg_MAE": eval_df_hr[eval_df_hr["Method"] == method]["MAE"].mean(),
    "Avg_PC": eval_df_hr[eval_df_hr["Method"] == method]["PC"].mean()
} for method in methods])

print("\nSummary DataFrame:")
summary_avg

Average MAE and PC across Rest and Task conditions:
Method     Avg MAE      Avg PC      
------------------------------------------------------------
POS        8.2917       0.636858    
GREEN      8.1042       0.551322    
LGI        6.3750       0.600468    
OMIT       6.4583       0.590413    
CHROM      7.8750       0.604770    

Summary DataFrame:


,Method,Avg_MAE,Avg_PC
0,POS,8.291667,0.636858
1,GREEN,8.104167,0.551322
2,LGI,6.375000,0.600468
3,OMIT,6.458333,0.590413
4,CHROM,7.875000,0.604770


## SDNN

In [50]:
rows = []

for subject in subjects:
    row = [subject]
    # Rest (T1) SDNN
    for method in ["PPG", "POS", "GREEN", "LGI", "OMIT", "CHROM"]:
        val = df_metrics.query("Subject == @subject and Task == 'Istirahat' and Method == @method")["SDNN"]
        row.append(val.values[0] if len(val) > 0 else None)

    # Task (T2) SDNN
    for method in ["PPG", "POS", "GREEN", "LGI", "OMIT", "CHROM"]:
        val = df_metrics.query("Subject == @subject and Task == 'Speech' and Method == @method")["SDNN"]
        row.append(val.values[0] if len(val) > 0 else None)


    # Task (T3) SDNN
    for method in ["PPG", "POS", "GREEN", "LGI", "OMIT", "CHROM"]:
        val = df_metrics.query("Subject == @subject and Task == 'Tugas Mental' and Method == @method")["SDNN"]
        row.append(val.values[0] if len(val) > 0 else None)

    rows.append(row)

df_sdnn = pd.DataFrame(rows, columns=columns)
print("SDNN (Satuan: ms)")
df_sdnn


SDNN (Satuan: ms)


Rest                            Speech                             \
   Subject   GT  POS GREEN  LGI OMIT CHROM     GT  POS GREEN  LGI OMIT CHROM   
0      s41   54   59   181   69   69    92    297  103   271  127  122   109   
1      s42  110   99   237  109  111   139    239  199   293  279  293   244   
2      s43   45   51   169  100  100    76    197  170   320  247  246   193   
3      s44  141  203   253  177  176   223    261  212   337  220  218   225   
4      s45  100   82   270  110  110   130    220  180   330  263  261   195   
5      s46   59   59    66   59   59    80    215  163   308  220  220   180   
6      s47   90   74   205  106  107   109    281  110   288  129  138   103   
7      s48  314   91   267  122  122   111    292  150   285  231  228   180   
8      s49   44   42   237   98   98    55    315  168   304  238  234   199   
9      s50   28   38   217   49   49    66    232  106   256  124  124   118   
10     s51   80   86   213   83   83   125    285  161   323  197  182   188   
11     s52  190   96   210  104  104   121    165  138   292  187  186   169   
12     s53  114  142   200  143  143   153    198  145   309  179  183   174   
13     s54  152   56   269  126  126    80    142  142   305  197  205   167   
14     s55  166  117   368  149  149   142    223  138   308  165  170   159   
15     s56  312  123   359  179  183   206    271  178   328  253  264   213   

   Task                             
     GT  POS GREEN  LGI OMIT CHROM  
0   264   98   264  133  132   128  
1   185  183   290  247  238   214  
2   178  149   307  194  186   159  
3   220  180   288  206  207   222  
4   115  149   301  229  233   209  
5   104  143   255  177  186   151  
6   281  109   322  193  188   129  
7   300  155   310  234  236   193  
8   313  154   273  213  216   156  
9    86   69   256   78   82   117  
10  139  133   343  159  159   179  
11  237  133   272  163  161   159  
12  265  134   268  135  132   166  
13  117  117   278  214  213   140  
14  130  130   300  182  185   202  
15  262  174   296  208  209   178

#### Evaluasi SDNN (MAE, RMSE dan PC)

---

- **MAE** dan **RMSE** digunakan untuk mengukur seberapa besar perbedaan nilai SDNN antara hasil estimasi rPPG dan referensi PPG.
- **PC (Pearson Correlation)** digunakan untuk melihat tren atau korelasi antara hasil pengukuran SDNN dari rPPG dan PPG.

In [51]:
methods = ["POS", "CHROM", "GREEN", "LGI", "OMIT"]
conditions = ["Rest", "Speech", "Task"]

eval_results = []

for cond in conditions:
    gt = df_sdnn[(cond, "GT")]  # ground truth for condition rest / task
    for method in methods:
        row=[method]
        pred = df_sdnn[(cond, method)]
        
        mae = mean_absolute_error(gt, pred)
        rmse = np.sqrt(np.mean((gt - pred) ** 2))
        pc, _ = pearsonr(gt, pred)
        
        eval_results.append({
            "Condition": cond,
            "Method": method,
            "MAE": mae,
            # "RMSE": rmse,
            "PC": pc
        })

eval_df_sdnn = pd.DataFrame(eval_results)
eval_df_sdnn

,Condition,Method,MAE,PC
0,Rest,POS,50.9375,0.453787
1,Rest,CHROM,53.5625,0.549925
2,Rest,GREEN,113.5000,0.620018
3,Rest,LGI,43.3750,0.668770
4,Rest,OMIT,43.1250,0.675890
5,Speech,POS,85.6250,0.014521
6,Speech,CHROM,67.8125,-0.040807
7,Speech,GREEN,69.5000,-0.099090
8,Speech,LGI,62.9375,-0.031817
9,Speech,OMIT,64.0625,-0.074347


In [52]:
# Calculate average MAE and PC for each method across conditions
methods = ["POS", "GREEN", "LGI", "OMIT", "CHROM"]

print("Average MAE and PC across Rest and Task conditions:")
print("=" * 60)
print(f"{'Method':<10} {'Avg MAE':<12} {'Avg PC':<12}")
print("-" * 60)

for method in methods:
    method_data = eval_df_sdnn[eval_df_sdnn["Method"] == method]
    avg_mae = method_data["MAE"].mean()
    avg_pc = method_data["PC"].mean()
    print(f"{method:<10} {avg_mae:<12.4f} {avg_pc:<12.6f}")

print("=" * 60)

# Create a summary DataFrame
summary_avg = pd.DataFrame([{
    "Method": method,
    "Avg_MAE": eval_df_sdnn[eval_df_sdnn["Method"] == method]["MAE"].mean(),
    "Avg_PC": eval_df_sdnn[eval_df_sdnn["Method"] == method]["PC"].mean()
} for method in methods])

print("\nSummary DataFrame:")
summary_avg

Average MAE and PC across Rest and Task conditions:
Method     Avg MAE      Avg PC      
------------------------------------------------------------
POS        69.1042      0.245675    
GREEN      92.3958      0.199813    
LGI        58.3333      0.282490    
OMIT       58.5833      0.261035    
CHROM      64.8333      0.164438    

Summary DataFrame:


,Method,Avg_MAE,Avg_PC
0,POS,69.104167,0.245675
1,GREEN,92.395833,0.199813
2,LGI,58.333333,0.282490
3,OMIT,58.583333,0.261035
4,CHROM,64.833333,0.164438


#### Ekstraksi HRV (RMSSD)

---

In [53]:
rows = []

for subject in subjects:
    row = [subject]
    # Rest (T1) RMSSD
    for method in ["PPG", "POS", "GREEN", "LGI", "OMIT", "CHROM"]:
        val = df_metrics.query("Subject == @subject and Task == 'Istirahat' and Method == @method")["RMSSD"]
        row.append(val.values[0] if len(val) > 0 else None)

    # Task (T2) RMSSD
    for method in ["PPG", "POS", "GREEN", "LGI", "OMIT", "CHROM"]:
        val = df_metrics.query("Subject == @subject and Task == 'Speech' and Method == @method")["RMSSD"]
        row.append(val.values[0] if len(val) > 0 else None)

     
    # Task (T3) RMSSD
    for method in ["PPG", "POS", "GREEN", "LGI", "OMIT", "CHROM"]:
        val = df_metrics.query("Subject == @subject and Task == 'Tugas Mental' and Method == @method")["RMSSD"]
        row.append(val.values[0] if len(val) > 0 else None)

    rows.append(row)

df_rmssd = pd.DataFrame(rows, columns=columns)
print("RMSSD (Satuan: ms)")
df_rmssd

RMSSD (Satuan: ms)


Rest                            Speech                             \
   Subject   GT  POS GREEN  LGI OMIT CHROM     GT  POS GREEN  LGI OMIT CHROM   
0      s41   64   64   240   86   85   125    353  131   365  162  160   138   
1      s42  128   88   296  123  117   195    343  266   395  388  394   305   
2      s43   53   68   239  145  141   113    235  219   397  330  329   257   
3      s44  149  212   316  198  195   303    332  285   469  270  260   283   
4      s45  145  116   349  149  149   188    294  261   414  376  376   274   
5      s46   58   62    75   60   60   102    278  214   404  318  318   234   
6      s47  104   69   273  132  134   148    370  139   370  173  186   124   
7      s48  362   90   382  158  157   161    376  205   351  316  310   255   
8      s49   47   39   346  135  135    65    406  224   396  301  302   259   
9      s50   20   46   311   64   64    95    312  123   364  163  162   146   
10     s51   91  100   244   86   86   160    401  202   451  266  238   244   
11     s52   99  107   283  125  125   161    180  190   385  239  238   230   
12     s53  157   87   243   93   93   129    236  166   406  214  221   234   
13     s54  187   80   357  185  185   109    190  191   397  256  264   237   
14     s55  207  140   452  198  184   193    306  178   397  215  215   193   
15     s56  440  171   458  252  256   274    391  231   442  349  363   287   

   Task                             
     GT  POS GREEN  LGI OMIT CHROM  
0   348  132   358  180  181   173  
1   235  239   407  331  320   286  
2   243  205   398  264  259   221  
3   283  215   362  247  249   257  
4   155  213   410  330  334   296  
5   116  178   313  240  246   212  
6   335  139   405  255  243   158  
7   426  195   379  316  323   236  
8   428  199   339  297  310   219  
9    97   91   343  108  111   163  
10  177  162   458  211  211   239  
11  311  165   367  220  215   224  
12  345  159   350  163  153   221  
13  154  164   391  290  288   199  
14  153  173   396  237  239   267  
15  353  238   405  270  269   236

In [54]:
methods = ["POS", "CHROM", "GREEN", "LGI", "OMIT"]
conditions = ["Rest", "Speech", "Task"]

eval_results = []

for cond in conditions:
    gt = df_rmssd[(cond, "GT")]  # ground truth for this condition
    for method in methods:
        row=[method]
        pred = df_rmssd[(cond, method)]
        
        mae = mean_absolute_error(gt, pred)
        rmse = np.sqrt(np.mean((gt - pred) ** 2))
        pc, _ = pearsonr(gt, pred)
        
        eval_results.append({
            "Condition": cond,
            "Method": method,
            "MAE": mae,
            # "RMSE": rmse,
            "PC": pc
        })

eval_df_rmssd = pd.DataFrame(eval_results)
eval_df_rmssd

,Condition,Method,MAE,PC
0,Rest,POS,63.8750,0.558860
1,Rest,CHROM,74.0000,0.585773
2,Rest,GREEN,159.5625,0.663630
3,Rest,LGI,52.0000,0.739284
4,Rest,OMIT,52.6875,0.752105
5,Speech,POS,112.5000,0.090448
6,Speech,CHROM,96.3125,-0.007052
7,Speech,GREEN,91.8750,0.118451
8,Speech,LGI,90.0625,0.094977
9,Speech,OMIT,91.5625,0.065124


Berdasarkan hasil ini, akan dibuat sebuah PoC dengan menggunakan metode POS / CHROM karena metode ini masih cukup tahan (robust) pada dua kondisi yang berbeda sehingga cocok untuk melakukan monitoring terhadap individu.